In [1]:
import argparse
import json
import os
import random as rn
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler, random_split

import torchvision
from torchvision import datasets, transforms
from torchvision.datasets import CIFAR10, MNIST
import torchvision.transforms.functional as F

import tensorboard
import tensorboardX

import nni
import nni.nas.strategy as strategy
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from nni.nas.nn.pytorch import (
    ModelSpace,
    LayerChoice,
    ValueChoice,
    MutableConv2d,
    MutableBatchNorm2d,
    MutableReLU,
    MutableLinear
)
from nni.nas.experiment import NasExperiment
from nni.nas.experiment.config import NasExperimentConfig
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator.pytorch import (
    Lightning,
    Classification,
    ClassificationModule
)

from nni.experiment.config import utils, ExperimentConfig

import pytorch_lightning
from pytorch_lightning import Trainer, LightningModule
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, array_to_img, load_img
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.applications import VGG19, VGG16, ResNet50
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import genotypes

torch.set_float32_matmul_precision('medium')


C:\Users\senti\anaconda3\envs\darts\Lib\site-packages\nni\nas\nn\pytorch\layers.py:94: RuntimeWarning: <class 'torch.nn.parameter.Buffer'> is found to be not a nn.Module, which is unexpected. It means your PyTorch version might not be supported.
  warnings.warn(f'{obj} is found to be not a nn.Module, which is unexpected. '


In [2]:
data_dir = 'C:/Users/senti/Desktop/GTSDB'
train_path = 'C:/Users/senti/Desktop/GTSDB/Train'
test_path = 'C:/Users/senti/Desktop/GTSDB/Test'
height = 32
width = 32

In [3]:
def check_dir(path):
    if not os.path.exists(path):
        print(f"NOT FOUND: {path}")
        return
    if not os.path.isdir(path):
        print(f"Not a directory: {path}")
        return
    if len(os.listdir(path)) == 0:
        print(f"EMPTY: {path}")
    else:
        print(f"OK: {path} ({len(os.listdir(path))} items)")

check_dir(data_dir)
check_dir(train_path)
check_dir(test_path)


✅ OK: C:/Users/senti/Desktop/GTSDB (5 items)
✅ OK: C:/Users/senti/Desktop/GTSDB/Train (46 items)
✅ OK: C:/Users/senti/Desktop/GTSDB/Test (303 items)


In [4]:
classes = { 0:'Speed limit (20km/h)',
            1:'Speed limit (30km/h)', 
            2:'Speed limit (50km/h)', 
            3:'Speed limit (60km/h)', 
            4:'Speed limit (70km/h)', 
            5:'Speed limit (80km/h)', 
            6:'End of speed limit (80km/h)', 
            7:'Speed limit (100km/h)', 
            8:'Speed limit (120km/h)', 
            9:'No passing', 
            10:'No passing veh over 3.5 tons', 
            11:'Right-of-way at intersection', 
            12:'Priority road', 
            13:'Yield', 
            14:'Stop', 
            15:'No vehicles', 
            16:'Veh > 3.5 tons prohibited', 
            17:'No entry', 
            18:'General caution', 
            19:'Dangerous curve left', 
            20:'Dangerous curve right', 
            21:'Double curve', 
            22:'Bumpy road', 
            23:'Slippery road', 
            24:'Road narrows on the right', 
            25:'Road work', 
            26:'Traffic signals', 
            27:'Pedestrians', 
            28:'Children crossing', 
            29:'Bicycles crossing', 
            30:'Beware of ice/snow',
            31:'Wild animals crossing', 
            32:'End speed + passing limits', 
            33:'Turn right ahead', 
            34:'Turn left ahead', 
            35:'Ahead only', 
            36:'Go straight or right', 
            37:'Go straight or left', 
            38:'Keep right', 
            39:'Keep left', 
            40:'Roundabout mandatory', 
            41:'End of no passing', 
            42:'End no passing veh > 3.5 tons' }

In [5]:
batch_size = 32

In [6]:
import os
import numpy as np
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
from sklearn.model_selection import StratifiedShuffleSplit

train_crop_root = r"C:/Users/senti/Desktop/GTSDB/Train"  # <-- cambia se serve

# Sanity check
print("Root exists:", os.path.exists(train_crop_root))
print("Subfolders:", sorted([d for d in os.listdir(train_crop_root) if os.path.isdir(os.path.join(train_crop_root, d))])[:10])

train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

val_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset_full = ImageFolder(root=train_crop_root, transform=train_transform)
val_dataset_full   = ImageFolder(root=train_crop_root, transform=val_transform)

print("Total samples:", len(train_dataset_full))
print("Num classes:", len(train_dataset_full.classes))
print("Classes head:", train_dataset_full.classes[:5], "...")

targets = np.array(train_dataset_full.targets)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))

train_subset = Subset(train_dataset_full, train_idx)
val_subset   = Subset(val_dataset_full,   val_idx)

print("Train subset:", len(train_subset))
print("Val subset:", len(val_subset))

val_targets = targets[val_idx]
missing_classes = set(range(len(train_dataset_full.classes))) - set(np.unique(val_targets))
print("Missing classes in val:", sorted(list(missing_classes))[:20], " ... total", len(missing_classes))
num_workers=8
batch_size = 32
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,   
    prefetch_factor=4          
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True
)


Root exists: True
Subfolders example: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09']
Total samples: 855
Num classes: 43
Classes head: ['00', '01', '02', '03', '04'] ...
Train subset: 684
Val subset: 171
Missing classes in val: [24, 37]  ... total 2


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)  


In [8]:
import torch
import nni

@nni.trace
class DartsClassificationModule(ClassificationModule):
    def __init__(
        self,
        learning_rate: float = 0.001,
        weight_decay: float = 5e-4,
        auxiliary_loss_weight: float = 0.4,
        max_epochs: int = 600,
        val_loader=None,                
    ):
        super().__init__(learning_rate=learning_rate, weight_decay=weight_decay,
                         export_onnx=False, num_classes=43)

        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate
        self._external_val_loader = val_loader

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate,
                                    momentum=0.9, weight_decay=0.)

        return {
            'optimizer': optimizer,
            'lr_scheduler': torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
        }

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()

        if self.auxiliary_loss_weight:
            y_hat, y_aux = self.model(x)   
            loss_main = self.criterion(y_hat, y)
            loss_aux  = self.criterion(y_aux, y)
            loss = loss_main + self.auxiliary_loss_weight * loss_aux

            self.log('train_loss_main', loss_main, on_step=False, on_epoch=True)
            self.log('train_loss_aux',  loss_aux,  on_step=False, on_epoch=True)
        else:
            y_hat = self.model(x)          # <<< usa self.model
            loss = self.criterion(y_hat, y)

        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)

        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y),
                     prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()

        y_hat = self.model(x)              
        loss = self.criterion(y_hat, y)

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)

        for name, metric in self.metrics.items():
            self.log("val_" + name, metric(y_hat, y),
                     prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def on_train_epoch_start(self):
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'],
                 on_step=False, on_epoch=True)

    def _get_val_loaders(self):
        if self._external_val_loader is not None:
            return [self._external_val_loader]

        dls = getattr(self.trainer, "val_dataloaders", None)
        if dls:
            return dls if isinstance(dls, (list, tuple)) else [dls]

        dm = getattr(self.trainer, "datamodule", None)
        if dm is not None and hasattr(dm, "val_dataloader"):
            dls = dm.val_dataloader()
            return dls if isinstance(dls, (list, tuple)) else [dls]

        return []

    def on_train_epoch_end(self):
        val_loaders = self._get_val_loaders()
        if not val_loaders:
            return

        was_training = self.model.training
        self.model.eval()

        correct, total = 0, 0
        loss_sum, n_batches = 0.0, 0

        with torch.no_grad():
            for loader in val_loaders:
                for x, y in loader:
                    x = x.to(self.device)
                    y = y.long().to(self.device)

                    logits = self.model(x)
                    loss = self.criterion(logits, y)

                    loss_sum += float(loss.item())
                    n_batches += 1

                    pred = logits.argmax(dim=1)
                    correct += int((pred == y).sum().item())
                    total += int(y.numel())

        val_loss = loss_sum / max(n_batches, 1)
        val_acc  = correct / max(total, 1)

        self.log("val_loss", val_loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc",  val_acc,  prog_bar=True, on_step=False, on_epoch=True)

        if was_training:
            self.model.train()


# DOF 2


class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=43, layers=7, verbose=0, drop_path_prob=0.1):
        super(CustomDARTSSpace, self).__init__()

        # ________________________________________________________________________________________________________________________
        # Initialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose

        # ________________________________________________________________________________________________________________________
        # Fixed channels (NO choices)
        layer0_out = 16
        layer1_out = 16
        layer2_out = 16
        layer3_out = 16
        layer4_out = 16
        layer5_out = 16
        layer6_out = 16
        layer7_out = 22  # fixed, so fc1 in_features will be 22*3*3 = 198

        # ________________________________________________________________________________________________________________________
        # Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)

        # ________________________________________________________________________________________________________________________
        # ONLY DOF = 2: a single LayerChoice with 2 options
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                nn.Conv2d(layer0_out, layer1_out, kernel_size=3, padding=0, bias=False),
                nn.BatchNorm2d(layer1_out),
                nn.ReLU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv2d(layer0_out, layer1_out, kernel_size=3, padding=0, bias=False),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                nn.BatchNorm2d(layer1_out),
                nn.ReLU(inplace=True)
            )
        ], label='layer_1')
        self.layers.append(layer1)

        # ________________________________________________________________________________________________________________________
        # Rest of the network FIXED (no LayerChoice, no channel choice)
        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            nn.Conv2d(layer1_out, layer2_out, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(layer2_out),
            nn.ReLU(inplace=True)
        ))

        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            nn.Conv2d(layer2_out, layer3_out, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(layer3_out),
            nn.ReLU(inplace=True)
        ))

        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            nn.Conv2d(layer3_out, layer4_out, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(layer4_out),
            nn.ReLU(inplace=True)
        ))

        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            nn.Conv2d(layer4_out, layer5_out, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(layer5_out),
            nn.ReLU(inplace=True)
        ))

        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            nn.Conv2d(layer5_out, layer6_out, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(layer6_out),
            nn.ReLU(inplace=True)
        ))

        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            nn.Conv2d(layer6_out, layer7_out, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(layer7_out),
            nn.ReLU(inplace=True)
        ))

        # ________________________________________________________________________________________________________________________
        # Linear (fixed; and compute fc1 input correctly)
        self.pool = nn.AdaptiveAvgPool2d((3, 3))

        feature1 = 32
        feature2 = 32
        feature3 = 32

        in_features = layer7_out * 3 * 3  # <-- no more mismatch
        self.fc1 = nn.Linear(in_features, feature1)
        self.fc2 = nn.Linear(feature1, feature2)
        self.fc3 = nn.Linear(feature2, feature3)
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(feature3, 43)

    def forward(self, x):
        # ________________________________________________________________________________________________________________________
        # Layer 0
        x = self.preliminary_layer(x)
        x = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1:
            print(f'After preliminary layer: {x.shape}')

        # ________________________________________________________________________________________________________________________
        # Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1:
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1:
                    print(f'After avg pooling: {x.shape}')

        # ________________________________________________________________________________________________________________________
        # Adaptive pool
        x = self.pool(x)
        if self.verbose == 1:
            print(f'After adaptive pooling: {x.shape}')

        # ________________________________________________________________________________________________________________________
        # Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1:
            print(f'After flattening: {x.shape}')

        # ________________________________________________________________________________________________________________________
        # FC stack
        x = self.fc1(x); x = self.relu(x)
        if self.verbose == 1:
            print(f'After fc1: {x.shape}')
        x = self.fc2(x); x = self.relu(x)
        if self.verbose == 1:
            print(f'After fc2: {x.shape}')
        x = self.fc3(x); x = self.relu(x)
        if self.verbose == 1:
            print(f'After fc3: {x.shape}')

        # ________________________________________________________________________________________________________________________
        # Classification
        x = self.classifier(x)
        if self.verbose == 1:
            print(f'After classifier: {x.shape}')
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


# DOF 10

class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=43, layers=7,verbose =0, drop_path_prob = 0.1):
        super(CustomDARTSSpace, self).__init__()

        #________________________________________________________________________________________________________________________
        #Inizialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose


        #________________________________________________________________________________________________________________________
        #Channel choices
        layer0_out = 16
        layer1_out = nni.choice('layer1_out_channels', [16,32,64])
        layer2_out= nni.choice('layer2_out_channels', [16,32,64])
        layer3_out = 16
        layer4_out = 16
        layer5_out = 16
        layer6_out = 16
        layer7_out = 22  # fixed, so fc1 in_features will be 22*3*3 = 198
        
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)
        
        #________________________________________________________________________________________________________________________
        #Layer 1
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1 ),
                MutableConv2d(layer0_out, layer1_out, kernel_size=3, bias=False),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer0_out, layer1_out, kernel_size=3, bias=False),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            )
        ], label='layer_1')
        self.layers.append(layer1)
        
        #________________________________________________________________________________________________________________________
        #Layer 2
        layer2 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer1_out, layer2_out, kernel_size=3, bias=False),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer1_out, layer2_out, kernel_size=3, bias=False),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            )
        ], label='layer_2')
        self.layers.append(layer2)
        
        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            MutableConv2d(layer2_out, layer3_out, kernel_size=3, padding=0, bias=False),
            MutableBatchNorm2d(layer3_out),
            MutableReLU()
        ))
        
        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            MutableConv2d(layer3_out, layer4_out, kernel_size=3, padding=0, bias=False),
            MutableBatchNorm2d(layer4_out),
            MutableReLU()
        ))
        
        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            MutableConv2d(layer4_out, layer5_out, kernel_size=3, padding=0, bias=False),
            MutableBatchNorm2d(layer5_out),
            MutableReLU()
        ))
        
        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            MutableConv2d(layer5_out, layer6_out, kernel_size=3, padding=0, bias=False),
            MutableBatchNorm2d(layer6_out),
            MutableReLU()
        ))
        
        self.layers.append(nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
            MutableConv2d(layer6_out, layer7_out, kernel_size=3, padding=0, bias=False),
            MutableBatchNorm2d(layer7_out),
            MutableReLU()
        ))
                

        
        #________________________________________________________________________________________________________________________
        #Linear
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        feature1 = 32
        feature2 = 32
        feature3 = 32
        self.fc1 = MutableLinear(198, feature1) 
        self.fc2 = MutableLinear(feature1, feature2) 
        self.fc3 = MutableLinear(feature2, feature3)  
        self.relu = nn.ReLU()
        self.classifier = MutableLinear(feature3, 43)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        #Layer 0
        x = self.preliminary_layer(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1 :
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1 :
                    print(f'After avg pooling: {x.shape}')
        
        #________________________________________________________________________________________________________________________
        #Adaprive pool
        x =  self.pool(x)
        if self.verbose == 1 :
            print(f'After adaptive pooling: {x.shape}')

        #________________________________________________________________________________________________________________________
        #Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        #________________________________________________________________________________________________________________________
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.fc2(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc2: {x.shape}')
        x = self.fc3(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc3: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Classification 
        x = self.classifier(x)

        
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


# DOF 26


class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=43, layers=7,verbose =0, drop_path_prob = 0.1):
        super(CustomDARTSSpace, self).__init__()

        #________________________________________________________________________________________________________________________
        #Inizialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose


        #________________________________________________________________________________________________________________________
        #Channel choices
        layer0_out = 16
        layer1_out = nni.choice('layer1_out_channels', [16,32,64])
        layer2_out= nni.choice('layer2_out_channels', [16,32,64])
        layer3_out= nni.choice('layer3_out_channels', [16,32,64])
        layer4_out = nni.choice('layer4_out_channels', [16,32,64])
        layer5_out = 16
        layer6_out = 16
        layer7_out = 22  # fixed, so fc1 in_features will be 22*3*3 = 198
        
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)
        

 #________________________________________________________________________________________________________________________
        #Layer 1
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1 ),
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            )
        ], label='layer_1')
        self.layers.append(layer1)
        
        #________________________________________________________________________________________________________________________
        #Layer 2
        layer2 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            )
        ], label='layer_2')
        self.layers.append(layer2)
        
        #________________________________________________________________________________________________________________________
        #Layer 3
        layer3 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            )
        ], label='layer_3')
        self.layers.append(layer3)
                #________________________________________________________________________________________________________________________
        #Layer 4
        layer4 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            )
        ], label='layer_4')
        self.layers.append(layer4)
                #________________________________________________________________________________________________________________________
        #Layer 5
        layer5 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            )
        ], label='layer_5')
        self.layers.append(layer5)
                #________________________________________________________________________________________________________________________
        #Layer 6
        layer6= LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            )
        ], label='layer_6')
        self.layers.append(layer6)
                #________________________________________________________________________________________________________________________
        #Layer 7
        layer7 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            )
        ], label='layer_7')
        self.layers.append(layer7)
                

        
        #________________________________________________________________________________________________________________________
        #Linear
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        feature1 = 32
        feature2 = 32
        feature3 = 32
        self.fc1 = MutableLinear(198, feature1) 
        self.fc2 = MutableLinear(feature1, feature2) 
        self.fc3 = MutableLinear(feature2, feature3)  
        self.relu = nn.ReLU()
        self.classifier = MutableLinear(feature3, 43)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        #Layer 0
        x = self.preliminary_layer(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1 :
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1 :
                    print(f'After avg pooling: {x.shape}')
        
        #________________________________________________________________________________________________________________________
        #Adaprive pool
        x =  self.pool(x)
        if self.verbose == 1 :
            print(f'After adaptive pooling: {x.shape}')

        #________________________________________________________________________________________________________________________
        #Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        #________________________________________________________________________________________________________________________
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.fc2(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc2: {x.shape}')
        x = self.fc3(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc3: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Classification 
        x = self.classifier(x)

        
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


# DOF 35

In [11]:
class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=43, layers=7,verbose =0, drop_path_prob = 0.1):
        super(CustomDARTSSpace, self).__init__()

        #________________________________________________________________________________________________________________________
        #Inizialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose


        #________________________________________________________________________________________________________________________
        #Channel choices
        layer0_out = 16
        layer1_out = nni.choice('layer1_out_channels', [16,32,64])
        layer2_out= nni.choice('layer2_out_channels', [16,32,64])
        layer3_out= nni.choice('layer3_out_channels', [16,32,64])
        layer4_out = nni.choice('layer4_out_channels', [16,32,64])
        layer5_out = nni.choice('layer5_out_channels', [16,32,64])
        layer6_out = nni.choice('layer6_out_channels', [16,32,64])
        layer7_out = 22  # fixed, so fc1 in_features will be 22*3*3 = 198
  
        
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)
        
 #________________________________________________________________________________________________________________________
        #Layer 1
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1 ),
                MutableConv2d(layer0_out, layer1_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer0_out, layer1_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            )
        ], label='layer_1')
        self.layers.append(layer1)
        
        #________________________________________________________________________________________________________________________
        #Layer 2
        layer2 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer1_out, layer2_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer1_out, layer2_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            )
        ], label='layer_2')
        self.layers.append(layer2)
        
        #________________________________________________________________________________________________________________________
        #Layer 3
        layer3 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer2_out, layer3_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer2_out, layer3_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            )
        ], label='layer_3')
        self.layers.append(layer3)
                #________________________________________________________________________________________________________________________
        #Layer 4
        layer4 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer3_out, layer4_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer3_out, layer4_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            )
        ], label='layer_4')
        self.layers.append(layer4)
                #________________________________________________________________________________________________________________________
        #Layer 5
        layer5 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer4_out, layer5_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer4_out, layer5_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            )
        ], label='layer_5')
        self.layers.append(layer5)
                #________________________________________________________________________________________________________________________
        #Layer 6
        layer6= LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer5_out, layer6_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer5_out, layer6_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            )
        ], label='layer_6')
        self.layers.append(layer6)
                #________________________________________________________________________________________________________________________
        #Layer 7
        layer7 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer6_out, layer7_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer6_out, layer7_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            )
        ], label='layer_7')
        self.layers.append(layer7)
                

        
        #________________________________________________________________________________________________________________________
        #Linear
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        feature1 = nni.choice('feature1', [32, 64, 128])
        feature2 = 32
        feature3 = 32
        self.fc1 = MutableLinear(198, feature1) 
        self.fc2 = MutableLinear(feature1, feature2) 
        self.fc3 = MutableLinear(feature2, feature3)  
        self.relu = nn.ReLU()
        self.classifier = MutableLinear(feature3, 43)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        #Layer 0
        x = self.preliminary_layer(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1 :
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1 :
                    print(f'After avg pooling: {x.shape}')
        
        #________________________________________________________________________________________________________________________
        #Adaprive pool
        x =  self.pool(x)
        if self.verbose == 1 :
            print(f'After adaptive pooling: {x.shape}')

        #________________________________________________________________________________________________________________________
        #Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        #________________________________________________________________________________________________________________________
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.fc2(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc2: {x.shape}')
        x = self.fc3(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc3: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Classification 
        x = self.classifier(x)

        
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


# DOF 39


class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=43, layers=7,verbose =0, drop_path_prob = 0.1):
        super(CustomDARTSSpace, self).__init__()

        #________________________________________________________________________________________________________________________
        #Inizialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose


        #________________________________________________________________________________________________________________________
        #Channel choices
        layer0_out = 16
        layer1_out = nni.choice('layer1_out_channels', [16,32,64])
        layer2_out= nni.choice('layer2_out_channels', [16,32,64])
        layer3_out= nni.choice('layer3_out_channels', [16,32,64])
        layer4_out = nni.choice('layer4_out_channels', [16,32,64])
        layer5_out= nni.choice('layer5_out_channels', [16,32,64])
        layer6_out= nni.choice('layer6_out_channels', [16,32,64])
        layer7_out= 22
        
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)
        
        #________________________________________________________________________________________________________________________
        #Layer 1
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1 ),
                MutableConv2d(layer0_out, layer1_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer0_out, layer1_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            )
        ], label='layer_1')
        self.layers.append(layer1)
        
        #________________________________________________________________________________________________________________________
        #Layer 2
        layer2 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer1_out, layer2_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer1_out, layer2_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            )
        ], label='layer_2')
        self.layers.append(layer2)
        
        #________________________________________________________________________________________________________________________
        #Layer 3
        layer3 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer2_out, layer3_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer2_out, layer3_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            )
        ], label='layer_3')
        self.layers.append(layer3)
                #________________________________________________________________________________________________________________________
        #Layer 4
        layer4 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer3_out, layer4_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer3_out, layer4_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            )
        ], label='layer_4')
        self.layers.append(layer4)
                #________________________________________________________________________________________________________________________
        #Layer 5
        layer5 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer4_out, layer5_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer4_out, layer5_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            )
        ], label='layer_5')
        self.layers.append(layer5)
                #________________________________________________________________________________________________________________________
        #Layer 6
        layer6= LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer5_out, layer6_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer5_out, layer6_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            )
        ], label='layer_6')
        self.layers.append(layer6)
                #________________________________________________________________________________________________________________________
        #Layer 7
        layer7 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer6_out, layer7_out, kernel_size=3, bias=True),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer6_out, layer7_out, kernel_size=3, bias=True),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            )
        ], label='layer_7')
        self.layers.append(layer7)
        

        
        #________________________________________________________________________________________________________________________
        #Linear
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        feature1 = nni.choice('feature1', [32, 64, 128])
        feature2 = nni.choice('feature2', [32 ,64, 128])
        feature3 = nni.choice('feature3', [32, 64])
        self.fc1 = MutableLinear(198, feature1) 
        self.fc2 = MutableLinear(feature1, feature2) 
        self.fc3 = MutableLinear(feature2, feature3)  
        self.relu = nn.ReLU()
        self.classifier = MutableLinear(feature3, 43)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        #Layer 0
        x = self.preliminary_layer(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1 :
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1 :
                    print(f'After avg pooling: {x.shape}')
        
        #________________________________________________________________________________________________________________________
        #Adaprive pool
        x =  self.pool(x)
        if self.verbose == 1 :
            print(f'After adaptive pooling: {x.shape}')

        #________________________________________________________________________________________________________________________
        #Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        #________________________________________________________________________________________________________________________
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.fc2(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc2: {x.shape}')
        x = self.fc3(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc3: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Classification 
        x = self.classifier(x)

        
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


# Execute DARTS SEARCH + val



In [ ]:
for i in range(5):
    checkpoint_callback = ModelCheckpoint(
        monitor="val_acc",
        dirpath="./checkpoints",
        filename="GTSDB-best-checkpoint",
        save_top_k=1,
        mode="max",
        save_on_train_epoch_end=True,
    )
    max_epochs = 200
    
    evaluator = Lightning(
        DartsClassificationModule(1e-2, 5e-4, 0., max_epochs, val_loader=val_loader),
        Trainer(
            accelerator="auto",
            callbacks=[checkpoint_callback],
            max_epochs=max_epochs,
            log_every_n_steps=1,
            precision="16-mixed"
        ),
        train_dataloaders=train_loader,
        val_dataloaders=val_loader
    )
    
    strategy = DartsStrategy(gradient_clip_val=0.)
    def search(log_dir: str, batch_size: int = 8):
    
        # Define model search space
        model_space = CustomDARTSSpace(input_channels=3, channels=64, num_classes=43, layers=7, verbose=0)
        model_space.set_drop_path_prob(0.)
    
        # Run NAS experiment
        exp_config = NasExperimentConfig.default(model_space, evaluator, strategy)
        exp_config.experiment_working_directory = "./DartsCheckpoints"
        exp_config.experiment_name = "Darts_search_DOF"
        exp_config.trial_concurrency = 1
        experiment = NasExperiment(model_space, evaluator, strategy, config = exp_config)
        experiment.run()
    
        return experiment
    experiment_results = search("./",32)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


[2026-01-24 16:08:29] Config is not provided. Will try to infer.
[2026-01-24 16:08:29] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:08:29] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ training_module │ DartsClassificationModule │  447 K │ train │     0 │
└───┴─────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 447 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 447 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 163                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

[2026-01-24 16:53:34] Config is not provided. Will try to infer.
[2026-01-24 16:53:34] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16:53:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-01-24 16

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ training_module │ DartsClassificationModule │  447 K │ train │     0 │
└───┴─────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 447 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 447 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 163                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()